In [1]:
import os
from pathlib import Path
from typing import Literal, Optional, List

import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white"})
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec
import seaborn as sns

os.environ["PROJECT_PATH"] = "/home/syebes/lambda-ana"
project_path = Path(os.environ["PROJECT_PATH"])
sys.path.append(str(project_path))
print(project_path/'data'/'df'/'test.df')

# My imports
from src.driver import AnalysisDriver
import src.branches as branches
from src.cuts import CutConfig, create_cut

/home/syebes/lambda-ana/data/df/test.df


# Load dfs

In [2]:
aurora_file = "/exp/sbnd/data/users/syebes/eafData/dfs/aurora_gen2.df"

In [3]:
with pd.HDFStore(aurora_file, 'r') as store:
    print(store.keys())

['/cuts', '/hdr', '/mcnu', '/pot']


In [4]:
aurora_hdr_df = pd.read_hdf(aurora_file, key='hdr')
aurora_mcnu_df = pd.read_hdf(aurora_file, key='mcnu') # Each entry is a nu interaction

In [5]:
aurora_mcnu_df.genie_mode

__ntuple  entry  rec.mc.nu..index
0         0      0                   10
                 1                    0
                 2                    0
                 3                   10
          1      0                   10
                                     ..
749261    12     1                    0
          13     0                    1
          14     0                    0
                 1                    0
                 2                   10
Name: genie_mode, Length: 17949418, dtype: int16

In [19]:
qe_lambda_mask = (aurora_mcnu_df.genie_mode == 0) & (aurora_mcnu_df.n_lmbda > 0) & (aurora_mcnu_df.pdg == -14)
qe_sigma0_mask = (aurora_mcnu_df.genie_mode == 0) & (aurora_mcnu_df.n_s0 > 0) & (aurora_mcnu_df.pdg == -14)
qe_lambda_signal_mask = aurora_mcnu_df.is_lambda_signal & (aurora_mcnu_df.pdg == -14)

In [20]:
in_tpc_mask = ((aurora_mcnu_df.position.x > -200) &
              (aurora_mcnu_df.position.x < 200) &
              (aurora_mcnu_df.position.y > -200) &
              (aurora_mcnu_df.position.y < 200) &
              (aurora_mcnu_df.position.z > 0) &
              (aurora_mcnu_df.position.x < 500))

In [24]:
sum(qe_sigma0_mask & qe_lambda_mask)

0

In [21]:
n_nu = len(aurora_mcnu_df)
n_lambda = len(aurora_mcnu_df[qe_lambda_mask])
n_sigma0 = len(aurora_mcnu_df[qe_sigma0_mask])
n_lambda_signal = len(aurora_mcnu_df[qe_lambda_signal_mask])
n_hyp = n_lambda + n_sigma0

print(f'# nu interactions: {n_nu}')
print(f'# qe lambda interactions: {n_lambda} ({100.*n_lambda/n_nu:.2e}%)')
print(f'# qe sigma0 interactions: {n_sigma0} ({100.*n_sigma0/n_nu:.2e}%)')
print(f'# qe neutral hyperon interactions: {n_hyp} ({100.*n_hyp/n_nu:.2e}%)')
print(f'\n# qe lambda signal interactions: {n_lambda_signal} ({100.*n_lambda_signal/n_lambda:.2f}% of qe lambdas)')

# nu interactions: 17949418
# qe lambda interactions: 2279 (1.27e-02%)
# qe sigma0 interactions: 878 (4.89e-03%)
# qe neutral hyperon interactions: 3157 (1.76e-02%)

# qe lambda signal interactions: 1428 (62.66% of qe lambdas)


In [22]:
n_nu = len(aurora_hdr_df)
n_lambda = len(aurora_mcnu_df[in_tpc_mask & qe_lambda_mask])
n_sigma0 = len(aurora_mcnu_df[in_tpc_mask & qe_sigma0_mask])
n_lambda_signal = len(aurora_mcnu_df[in_tpc_mask & qe_lambda_signal_mask])
n_hyp = n_lambda + n_sigma0

print(f'# nu interactions: {n_nu}')
print(f'# qe lambda interactions: {n_lambda} ({1.*n_lambda/n_nu:.2e})')
print(f'# qe sigma0 interactions: {n_sigma0} ({1.*n_sigma0/n_nu:.2e})')
print(f'# qe neutral hyperon interactions: {n_hyp} ({1.*n_hyp/n_nu:.2e})')
print(f'\n# qe lambda signal interactions: {n_lambda_signal} ({1.*n_lambda_signal/n_lambda:.2f} of qe lambdas)')

# nu interactions: 9935461
# qe lambda interactions: 560 (5.64e-05)
# qe sigma0 interactions: 225 (2.26e-05)
# qe neutral hyperon interactions: 785 (7.90e-05)

# qe lambda signal interactions: 358 (0.64 of qe lambdas)


In [18]:
n_nu = len(aurora_hdr_df)
n_lambda = len(aurora_mcnu_df[in_tpc_mask & qe_lambda_mask].index.droplevel(-1).unique())
n_sigma0 = len(aurora_mcnu_df[in_tpc_mask & qe_sigma0_mask].index.droplevel(-1).unique())
n_lambda_signal = len(aurora_mcnu_df[in_tpc_mask & qe_lambda_signal_mask])
n_hyp = n_lambda + n_sigma0

print(f'# nu events: {n_nu}')
print(f'# qe lambda interactions: {n_lambda} ({1.*n_lambda/n_nu:.2e})')
print(f'# qe sigma0 interactions: {n_sigma0} ({1.*n_sigma0/n_nu:.2e})')
print(f'# qe neutral hyperon interactions: {n_hyp} ({1.*n_hyp/n_nu:.2e})')
print(f'\n# qe lambda signal interactions: {n_lambda_signal} ({1.*n_lambda_signal/n_lambda:.2f} of qe lambdas)')

# nu events: 9935461
# qe lambda interactions: 560 (5.64e-05)
# qe sigma0 interactions: 225 (2.26e-05)
# qe neutral hyperon interactions: 785 (7.90e-05)

# qe lambda signal interactions: 358 (0.64 of qe lambdas)


In [ ]:
len(aurora_mcnu_df.index.droplevel(-1).unique())

In [ ]:
aurora_mcnu_df.genie_mode.value_counts()

## Check if there is dirt

In [ ]:
aurora_mcnu_df.position.x
aurora_mcnu_df.position.y
aurora_mcnu_df.position.z

In [ ]:
len(aurora_mcnu_df)

In [ ]:
sum(aurora_mcnu_df.position.y < -200)